# Analyze Xenium data

The 10x Genomics Xenium In Situ platform is an image-based *in situ* technology that reports the counts of a pre-designed gene panel at single-cell resolution. An experiment outputs a single stitched tissue region in microns, with per-cell centroids, transcript locations, and multi-channel morphology images. Xenium is widely used for tumor microenvironment studies, brain atlasing, and validation of panel-design hypotheses from dissociated single-cell data.

This tutorial mirrors the NanoString CosMx tutorial (`t_nanostring_preprocess.ipynb`) but uses `ov.io.read_xenium()` and a standard Leiden-based downstream analysis. It is intentionally kept lightweight: we download a publicly available Xenium FFPE Human Breast Cancer sample directly from 10x's CDN and run the whole pipeline on the gene-expression counts without the full morphology image (which is several hundred MB).

What you will see:
- `ov.io.read_xenium()` parsing the 10x `outs/` layout into an `AnnData` with spatial coordinates in microns
- a drop of non-Gene-Expression features (control probes and codewords)
- the standard OmicVerse preprocessing pipeline (`normalize_total` → `log1p` → `scale` → `pca`)
- a neighbor graph over the PCA space and Leiden clustering
- spatial visualization of the clusters on the tissue layout

## 1. Environment setup

Import `omicverse`, set the plotting style, and enable auto-reload. `ov.style(font_path='Arial')` keeps exported figures consistent across platforms.

In [ ]:
from pathlib import Path
import numpy as np
import omicverse as ov
ov.style(font_path='Arial')

%load_ext autoreload
%autoreload 2

Initialize the CPU/GPU mixed runtime. Falls back to CPU automatically when no GPU is available.

In [ ]:
ov.settings.cpu_gpu_mixed_init()

## 2. Download the Xenium dataset

We use the 10x public **Xenium FFPE Human Breast Cancer Replicate 1** sample (≈167k cells, 313 genes, 280-gene Breast Cancer Tumor Microenvironment panel plus 33 custom targets). The full outs bundle is several GB because of the multi-channel morphology OME-TIFFs, but the minimum files needed for a Leiden analysis are just ≈30 MB:

- `cell_feature_matrix.h5` — sparse gene × cell counts
- `cells.csv.gz` — per-cell metadata with `x_centroid`, `y_centroid` in microns
- `experiment.xenium` — run metadata (panel, pixel size, etc.)

If you want the H&E / DAPI background overlay, also download `morphology_focus.ome.tif` (≈624 MB) and call `ov.io.read_xenium(..., load_image=True)`.

In [ ]:
# !mkdir -p data/xenium_breast_rep1
# BASE = 'https://cf.10xgenomics.com/samples/xenium/1.0.1/Xenium_FFPE_Human_Breast_Cancer_Rep1'
# !wget -O data/xenium_breast_rep1/cell_feature_matrix.h5   $BASE/Xenium_FFPE_Human_Breast_Cancer_Rep1_cell_feature_matrix.h5
# !wget -O data/xenium_breast_rep1/cells.csv.gz             $BASE/Xenium_FFPE_Human_Breast_Cancer_Rep1_cells.csv.gz
# !wget -O data/xenium_breast_rep1/experiment.xenium        $BASE/Xenium_FFPE_Human_Breast_Cancer_Rep1_experiment.xenium
# Optional — ~624 MB, only needed for the H&E overlay in ov.pl.spatial:
# !wget -O data/xenium_breast_rep1/morphology_focus.ome.tif $BASE/Xenium_FFPE_Human_Breast_Cancer_Rep1_morphology_focus.ome.tif

In [ ]:
sample_dir = Path('data') / 'xenium_breast_rep1'
ov.utils.print_tree(sample_dir)

## 3. Read the Xenium dataset

`ov.io.read_xenium()` handles the 10x `outs/` layout:

- reads `cell_feature_matrix.h5` via the 10x HDF5 parser
- drops non-Gene-Expression features (control probes / codewords) so downstream PCA and HVG don't waste capacity on them
- merges `cells.csv.gz` (or `cells.parquet`) into `obs`
- writes cell centroids in **microns** to `obsm['spatial']`
- populates `uns['spatial'][library_id]['scalefactors']` so that `ov.pl.spatial` can convert micron coords to image-pixel space when the morphology image is available
- parses `experiment.xenium` into `uns['spatial'][library_id]['metadata']`

Pass `load_image=False` to skip the (large) morphology OME-TIFF. Set `load_image=True` once you have `morphology_focus.ome.tif` next to `cells.csv.gz`.

In [ ]:
adata = ov.io.read_xenium(sample_dir, load_image=False)
adata

Write out a cached h5ad copy so re-opening is instant. Xenium `x_centroid` / `y_centroid` are in microns, not pixels.

In [ ]:
library_id = next(iter(adata.uns['spatial']))
sf = adata.uns['spatial'][library_id]['scalefactors']
print('library_id        :', library_id)
print('spatial range (µm):',
      adata.obsm['spatial'].min(axis=0).tolist(), '->',
      adata.obsm['spatial'].max(axis=0).tolist())
print('pixel size (µm/px):', 1 / sf['tissue_hires_scalef'])
print('mean cell diameter:', sf['spot_diameter_fullres'] / sf['tissue_hires_scalef'], 'µm')
adata.write('data/xenium_breast_rep1.h5ad')

## 4. Inspect spatial layout and per-cell QC

Quick sanity check: plot the total UMI per cell over the tissue layout. Xenium panel sizes are small (here 313 genes) so `total_counts` typically sits in the tens to hundreds.

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
ov.pl.embedding(
    adata, basis='spatial', color='total_counts',
    vmax='p99', cmap='Reds', ax=axs[0], show=False, title='total_counts',
)
axs[0].invert_yaxis()  # match image-coordinate convention
ov.pl.embedding(
    adata, basis='spatial', color='cell_area',
    vmax='p99', cmap='viridis', ax=axs[1], show=False, title='cell_area (µm²)',
)
axs[1].invert_yaxis()
plt.tight_layout()

## 5. Cell-level QC

For Xenium we typically filter out cells with very low transcript counts (segmentation artifacts or empty nuclei). A lower bound of 10 counts is a mild threshold that keeps most of the tissue intact; tune based on the `total_counts` histogram.

In [ ]:
import scipy.sparse as sp
counts = np.asarray(adata.X.sum(axis=1)).ravel() if sp.issparse(adata.X) else adata.X.sum(axis=1)
print(f'cells pre-QC : {adata.n_obs}')
adata = adata[counts >= 10].copy()
print(f'cells post-QC: {adata.n_obs} (>= 10 transcripts/cell)')

## 6. Normalize and log-transform

We use the standard `normalize_total` + `log1p` pipeline. Because the panel is small, we skip HVG selection and use all genes for PCA.

In [ ]:
ov.pp.normalize_total(adata, target_sum=1e4)
ov.pp.log1p(adata)

## 7. Scale and PCA

Z-score the log-normalized matrix, then compute the first 50 principal components. These become the reduced-dimensional representation used downstream for neighbor graph construction and Leiden clustering.

In [ ]:
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=50)

## 8. Neighbors + Leiden

Build a k-NN graph from the PCA embedding and run Leiden. For a 167k-cell Xenium section, `resolution=0.5` typically gives 10-20 clusters which map reasonably well to tissue structure. Tune `resolution` higher for finer subtypes, lower for broad regions.

In [ ]:
ov.pp.neighbors(
    adata, n_neighbors=15,
    use_rep='scaled|original|X_pca', n_pcs=50,
)
ov.pp.leiden(adata, resolution=0.5)
print(f"leiden: {adata.obs['leiden'].nunique()} clusters")

## 9. Visualize Leiden clusters on the tissue

Plot the Leiden label and one biological marker over the spatial layout. In a Breast Cancer section you typically see spatially-segregated clusters for epithelial / tumor tiles vs. stromal regions vs. lymphocyte infiltrates.

Note on orientation: Xenium centroids come in image-pixel convention (y grows downward). Calling `ax.invert_yaxis()` matches the tissue image when you overlay one later.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 6))
ov.pl.embedding(
    adata, basis='spatial', color='leiden',
    palette=ov.pl.palette_112,
    legend_fontsize=8, ax=axs[0], show=False, title='Leiden clusters',
)
axs[0].invert_yaxis()

# Pick a marker likely to be present in the Breast Cancer panel
marker = next((g for g in ['KRT7', 'EPCAM', 'ERBB2', 'ESR1', 'KRT14']
               if g in adata.var_names), adata.var_names[0])
ov.pl.embedding(
    adata, basis='spatial', color=marker,
    vmax='p99.2', cmap='Reds',
    ax=axs[1], show=False, title=marker,
)
axs[1].invert_yaxis()
plt.tight_layout()

## 10. (Optional) overlay on the morphology image

If you downloaded `morphology_focus.ome.tif` into `sample_dir`, re-read the dataset with `load_image=True` and use `ov.pl.spatial` to overlay the Leiden labels on the DAPI / morphology background. `read_xenium()` sets `tissue_hires_scalef = 1 / pixel_size` so the spot coordinates land correctly in image-pixel space.

```python
adata_img = ov.io.read_xenium(sample_dir, load_image=True)
adata_img.obs['leiden'] = adata.obs['leiden']   # carry over labels
ov.pl.spatial(
    adata_img, color='leiden', library_id=library_id,
    size=1.0, alpha_img=0.5, palette=ov.pl.palette_112,
)
```

For large Xenium sections (cm-scale), prefer `crop_coord=(x0, x1, y0, y1)` in image pixels to zoom into a region instead of rendering the full tissue.

## 11. Save the processed object

Persist the analyzed AnnData so you can skip preprocessing next time.

In [ ]:
adata.write('data/xenium_breast_rep1_processed.h5ad')